# Stage 2 Notebook 28 - Exp2W Query head + dense lane mask supervision

**Why this exists.** Exp2U (NB26 lane-only diagnostic) decisively ruled out detection-as-saboteur: removing detection completely had near-zero effect on lane metrics. The lane impasse is *intrinsic* to the lane task at our current configuration.

Re-examining 22 experiments with this lens reveals an important pattern:
- **Exp2N** (priors, `mask_aux: true, w_mask: 1.0`): matched_iou=0.42 -- geometry champion.
- **Every query-style experiment** (Exp2P/Q/R/S/T/U): `mask_aux: false, w_mask: 0`. matched_iou collapsed to 0.13.

**This is not a coincidence.** Across the modern lane detection literature -- CLRNet, CLRKDNet, CondLaneNet, BezierLaneNet -- a per-pixel lane segmentation auxiliary head is standard practice. It gives the backbone explicit dense 'where are lanes' supervision that no per-prior cls or per-curve regression can provide.

**We've been training the backbone without any pixel-level lane signal in every query-style experiment.** Exp2W is the simplest possible test of restoring it.

Single-knob change vs Exp2P:
- `lane_head.mask_aux: false -> true` (LaneQueryHead's existing mask_decoder branches off the largest backbone feature)
- `loss.lane.w_mask: 0.0 -> 2.0` (CLRNet uses 1.0; we go higher to make this dominate gradient magnitude since it's the missing piece)

All other architecture and losses identical to Exp2P. The dataset already produces `mask_target` from GT polylines via `soft_polyline_mask_numpy`; FusionLaneLoss already handles `mask_logit + mask_target` with BCE + Dice.

Reference: CLRNet (CVPR 2022), CLRKDNet (TIP 2024), CondLaneNet (ICCV 2021).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp23_rmt_gca_query_with_mask_aux_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp23_rmt_gca_query_with_mask_aux_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp23_rmt_gca_query_with_mask_aux_joint_smoke.log
OK exp23_rmt_gca_query_with_mask_aux_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=2.8105 det_loss=3.1758 grad_cos=-0.0915 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4959985911846161, 'gate/lane_mean': 0.5012624859809875, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp23_rmt_gca_query_with_mask_aux_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp23_rmt_gca_query_with_mask_aux_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp23_rmt_gca_query_with_mask_aux_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp23_rmt_gca_query_with_mask_aux_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp23_rmt_gca_query_with_mask_aux_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp23_rmt_gca_query_with_mask_aux_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp23_rmt_gca_query_with_mask_aux_joint.yaml --curve-tar /content/drive/MyDrive/EcoCA

0

## What to watch in Exp2W training

Reference Exp2P (queries, no mask): `decoded_f1=0.026`, `matched_iou=0.13`, `oracle_f1=0.07`.
Reference Exp2N (priors, with mask): `matched_iou=0.42`, `oracle_f1=0.27`.

Strong signals that mask supervision unblocks the impasse:
- **`val/lane/decoded_f1 >= 0.10`**: 4x current best. The metric that matters for CLRKDNet comparison.
- **`val/matched_line_iou >= 0.30`**: geometry should improve substantially because the backbone now has lane-pixel features. Predicted: closer to Exp2N's 0.42.
- **`val/lane/decoded_oracle_f1 >= 0.20`**: oracle ranking on better geometry should lift the ceiling.
- **`val_lane_f1 >= 0.55`**: cls task unchanged.
- **`val/lane/mask_aux` should decrease over training** (training signal is real, not a no-op).

Failure signals:
- decoded_f1 stays at ~0.03: mask supervision alone isn't enough; pivot to KD from teacher or extended training.
- Geometry improves but cls regresses: mask loss is competing with cls; reduce w_mask to 1.0.
- Mask loss diverges or stays flat: mask_target rendering may be broken; inspect a sample mask.